In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/Matches.csv")

/var/folders/28/lbx47lx91ds2yl8gz5v4g2380000gn/T/ipykernel_79591/2189308214.py:1: DtypeWarning: Columns (0: MatchTime) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/Matches.csv")


In [3]:
filtered_df = df[df['Division'] == "SP1"].copy()
filtered_df.shape

(9419, 48)

In [4]:

filtered_df['MatchDate'] = pd.to_datetime(filtered_df['MatchDate'])
filtered_df.shape


(9419, 48)

In [5]:
recent = filtered_df[filtered_df['MatchDate'] >= '2022-08-01'].copy()
recent.shape

(1551, 48)

In [6]:
recent['HomeTeam'].unique()

<StringArray>
[    'Osasuna',       'Celta',  'Valladolid',   'Barcelona',       'Cadiz',
    'Valencia',     'Almeria',  'Ath Bilbao',      'Getafe',       'Betis',
     'Espanol',     'Sevilla',    'Mallorca',  'Ath Madrid',    'Sociedad',
       'Elche',      'Girona',   'Vallecano', 'Real Madrid',  'Villarreal',
  'Las Palmas',      'Alaves',     'Granada',     'Leganes',     'Levante',
      'Oviedo',   'Santander',   'La Coruna',      'Malaga']
Length: 29, dtype: str

In [7]:
recent['FTHome'].isna().sum()

np.int64(0)

In [8]:
home_avg = recent['FTHome'].mean()
print("home avg:" , home_avg)

home avg: 1.4958091553836235


In [9]:
away_avg = recent['FTAway'].mean()
print("away avg:" , away_avg)

away avg: 1.1263700838168924


In [10]:
barca_home = recent[recent['HomeTeam'] == "Barcelona"]
barca_home_scored = barca_home['FTHome'].mean()
barca_home_attack = barca_home_scored / home_avg

print("home games:" , len(barca_home))
print("avg scored at home:" , barca_home_scored)
print("home attack strength:" , barca_home_attack)

home games: 78
avg scored at home: 2.5128205128205128
home attack strength: 1.6799071618037136


In [11]:
recent.groupby('HomeTeam')['FTHome'].mean()

HomeTeam
Alaves         1.118644
Almeria        1.342105
Ath Bilbao     1.545455
Ath Madrid     2.153846
Barcelona      2.512821
Betis          1.558442
Cadiz          0.921053
Celta          1.397436
Elche          1.230769
Espanol        1.271186
Getafe         0.974026
Girona         1.789474
Granada        1.263158
La Coruna      2.000000
Las Palmas     1.078947
Leganes        1.210526
Levante        1.550000
Malaga         1.000000
Mallorca       1.184211
Osasuna        1.384615
Oviedo         0.473684
Real Madrid    2.435897
Santander      2.500000
Sevilla        1.217949
Sociedad       1.423077
Valencia       1.282051
Valladolid     0.842105
Vallecano      1.233766
Villarreal     2.144737
Name: FTHome, dtype: float64

In [12]:
home_attack  = recent.groupby('HomeTeam')['FTHome'].mean() / home_avg
home_defense = recent.groupby('HomeTeam')['FTAway'].mean() / away_avg
away_attack  = recent.groupby('AwayTeam')['FTAway'].mean() / away_avg
away_defense = recent.groupby('AwayTeam')['FTHome'].mean() / home_avg

ratings = pd.DataFrame({
    'home_attack': home_attack,
    'home_defense': home_defense,
    'away_attack': away_attack,
    'away_defense': away_defense
})

ratings.head(29)

,home_attack,home_defense,away_attack,away_defense
Alaves,0.747852,0.917903,0.872501,1.037381
Almeria,0.897244,1.355075,0.957898,1.442627
Ath Bilbao,1.033190,0.807098,1.013011,0.874237
Ath Madrid,1.439920,0.808133,1.256767,0.711946
Barcelona,1.679907,0.648783,1.867855,0.729310
Betis,1.041872,0.887808,1.069922,0.959947
Cadiz,0.615755,0.981261,0.490631,1.161139
Celta,0.934234,1.104069,1.058540,1.045656
Elche,0.822812,1.206508,0.754637,1.337069
Espanol,0.849832,1.218855,0.994957,1.129593


In [13]:
ratings.isna().sum()

home_attack     0
home_defense    0
away_attack     0
away_defense    0
dtype: int64

In [14]:
import os
# test api key
print(bool(os.environ.get("FOOTBALL_API_KEY")))

True


In [15]:
import os, requests, json
resp = requests.get("https://api.football-data.org/v4/competitions/PD/matches",
    headers={"X-Auth-Token": os.environ.get("FOOTBALL_API_KEY")}
)
resp.raise_for_status()
data = resp.json()

with open("../data/raw/fixtures_PD.json", "w") as f:
    json.dump(data, f)

print(data["resultSet"])
print(len(data["matches"]), "matches")

{'count': 380, 'first': '2026-08-15', 'last': '2027-05-30', 'played': 41}
380 matches


In [16]:
m = data["matches"][0]
print(json.dumps(m, indent=2)[:1500])

{
  "area": {
    "id": 2224,
    "name": "Spain",
    "code": "ESP",
    "flag": "https://crests.football-data.org/760.svg"
  },
  "competition": {
    "id": 2014,
    "name": "Primera Division",
    "code": "PD",
    "type": "LEAGUE",
    "emblem": "https://crests.football-data.org/laliga.png"
  },
  "season": {
    "id": 2518,
    "startDate": "2026-08-16",
    "endDate": "2027-05-30",
    "currentMatchday": 6,
    "winner": null
  },
  "id": 564634,
  "utcDate": "2026-08-15T17:30:00Z",
  "status": "FINISHED",
  "matchday": 1,
  "stage": "REGULAR_SEASON",
  "group": null,
  "lastUpdated": "2026-09-09T00:20:32Z",
  "homeTeam": {
    "id": 263,
    "name": "Deportivo Alav\u00e9s",
    "shortName": "Alav\u00e9s",
    "tla": "ALA",
    "crest": "https://crests.football-data.org/263.png"
  },
  "awayTeam": {
    "id": 82,
    "name": "Getafe CF",
    "shortName": "Getafe",
    "tla": "GET",
    "crest": "https://crests.football-data.org/82.png"
  },
  "score": {
    "winner": "HOME_TEAM"

In [17]:
fixtures = pd.DataFrame([
    {
        "matchday": m["matchday"],
        "utc": m["utcDate"],
        "status": m["status"],
        "home": m["homeTeam"]["name"],
        "away": m["awayTeam"]["name"],
        "fh": m["score"]["fullTime"]["home"],
        "fa": m["score"]["fullTime"]["away"],
    }
    for m in data["matches"]
])
fixtures["utc"] = pd.to_datetime(fixtures["utc"])
fixtures.shape, fixtures["status"].value_counts()

((380, 7),
 status
 SCHEDULED    310
 FINISHED      41
 TIMED         29
 Name: count, dtype: int64)

In [18]:
api_names = sorted(set(fixtures["home"]) | set(fixtures["away"]))
print(len(api_names))
for n in api_names:
    print(repr(n))

print(sorted(ratings.index.tolist()))

20
'Athletic Club'
'CA Osasuna'
'Club Atlético de Madrid'
'Deportivo Alavés'
'Elche CF'
'FC Barcelona'
'Getafe CF'
'Levante UD'
'Málaga CF'
'RC Celta de Vigo'
'RC Deportivo La Coruña'
'RCD Espanyol de Barcelona'
'Rayo Vallecano de Madrid'
'Real Betis Balompié'
'Real Madrid CF'
'Real Racing Club de Santander'
'Real Sociedad de Fútbol'
'Sevilla FC'
'Valencia CF'
'Villarreal CF'
['Alaves', 'Almeria', 'Ath Bilbao', 'Ath Madrid', 'Barcelona', 'Betis', 'Cadiz', 'Celta', 'Elche', 'Espanol', 'Getafe', 'Girona', 'Granada', 'La Coruna', 'Las Palmas', 'Leganes', 'Levante', 'Malaga', 'Mallorca', 'Osasuna', 'Oviedo', 'Real Madrid', 'Santander', 'Sevilla', 'Sociedad', 'Valencia', 'Valladolid', 'Vallecano', 'Villarreal']


In [19]:
NAME_MAP = {
    "Athletic Club": "Ath Bilbao",
    "CA Osasuna": "Osasuna",
    "Club Atlético de Madrid": "Ath Madrid",
    "Deportivo Alavés": "Alaves",
    "Elche CF": "Elche",
    "FC Barcelona": "Barcelona",
    "Getafe CF": "Getafe",
    "Levante UD": "Levante",
    "Málaga CF": "Malaga",
    "RC Celta de Vigo": "Celta",
    "RC Deportivo La Coruña": "La Coruna",
    "Real Betis Balompié": "Betis",
    "Real Madrid CF": "Real Madrid",
    "Real Sociedad de Fútbol": "Sociedad",
    "Rayo Vallecano de Madrid": "Vallecano",
    "RCD Espanyol de Barcelona": "Espanol",
    "Real Racing Club de Santander": "Santander",
    "Sevilla FC": "Sevilla",
    "Valencia CF": "Valencia",
    "Villarreal CF": "Villarreal",
}

fixtures["home_r"] = fixtures["home"].map(NAME_MAP)
fixtures["away_r"] = fixtures["away"].map(NAME_MAP)


unmapped = (
    set(fixtures.loc[fixtures["home_r"].isna(), "home"]) |
    set(fixtures.loc[fixtures["away_r"].isna(), "away"])
)
print("unmapped teams:", unmapped)
assert not (set(NAME_MAP.values()) - set(ratings.index))
assert fixtures[["home_r", "away_r"]].notna().all().all(), "unmapped team"

unmapped teams: set()


In [20]:
from collections import Counter

played = Counter(recent["HomeTeam"]) + Counter(recent["AwayTeam"])
for t in sorted(set(NAME_MAP.values())):
    n = played.get(t, 0)
    flag = "⚠️" if n < 10 else ""
    print(f"{t:14} {played.get(t,0)}")

Alaves         117
Ath Bilbao     155
Ath Madrid     155
Barcelona      155
Betis          155
Celta          156
Elche          79
Espanol        117
Getafe         155
La Coruna      3
Levante        41
Malaga         3
Osasuna        155
Real Madrid    155
Santander      3
Sevilla        155
Sociedad       156
Valencia       155
Vallecano      155
Villarreal     155


In [21]:
k = 10
n = pd.Series(played).reindex(ratings.index).fillna(0)    # match count per rated team
w = n / (n + k)                                          # weighted on raw rating 0-1 because of minimal data
ratings_adj = ratings.mul(w, axis=0).add(1 - w, axis=0)

print(ratings_adj.loc[["Barcelona", "Malaga", "Santander", "La Coruna"]])
print(ratings_adj.describe())

           home_attack  home_defense  away_attack  away_defense
Barcelona     1.638701      0.670068     1.815258      0.745716
Malaga        0.923508      0.974109     0.769231      1.232062
Santander     1.154924      1.178988     0.769231      0.923508
La Coruna     1.077785      0.974109     0.974109      0.923508
       home_attack  home_defense  away_attack  away_defense
count    29.000000     29.000000    29.000000     29.000000
mean      0.962424      1.037801     0.945391      1.052567
std       0.275120      0.186356     0.260357      0.221752
min       0.459034      0.670068     0.549860      0.680467
25%       0.825499      0.894607     0.776973      0.923508
50%       0.909192      1.035413     0.882540      1.033571
75%       1.039334      1.183305     1.048181      1.119389
max       1.638701      1.392077     1.815258      1.608821


In [22]:
def expected_goals(home, away, R = ratings_adj):
    eh = home_avg * R.loc[home, "home_attack"] * R.loc[away, "away_defense"]
    ea = away_avg * R.loc[away, "away_attack"] * R.loc[home, "home_defense"]
    return eh, ea

print(expected_goals("Barcelona", "Real Madrid"))
print(expected_goals("Real Madrid", "Barcelona"))

(np.float64(1.6679500294272036), np.float64(1.1656872815596115))
(np.float64(1.7739999609402295), np.float64(1.457505153585822))


In [23]:
eg = fixtures.apply(lambda r: expected_goals(r["home_r"], r["away_r"]), axis=1, result_type="expand")
fixtures[["xg_home", "xg_away"]] = eg
fixtures.loc[fixtures["status"] == "FINISHED", ["home", "away", "matchday", "xg_home", "xg_away"]].head(12)

,home,away,matchday,xg_home,xg_away
0,Deportivo Alavés,Getafe CF,1,1.021922,0.820120
1,Sevilla FC,Rayo Vallecano de Madrid,1,1.148561,0.997998
2,Real Racing Club de Santander,Villarreal CF,1,1.629093,1.594581
3,RCD Espanyol de Barcelona,Levante UD,1,1.440213,1.231396
4,RC Deportivo La Coruña,Elche CF,1,2.094513,0.858242
5,Club Atlético de Madrid,Málaga CF,1,2.604537,0.710272
6,Rayo Vallecano de Madrid,Deportivo Alavés,2,1.292683,1.029269
7,Real Betis Balompié,Real Sociedad de Fútbol,2,1.333341,1.009202
8,Athletic Club,Sevilla FC,2,1.653438,0.994876
9,Valencia CF,RC Celta de Vigo,2,1.350570,1.050387
